In [39]:
import pandas as pd

# -------- FILE PATHS --------
file_2 = "C:\\Users\\praga\\Documents\\aXtr\\aXtr-electionCampagin\\genai-hyper-personalized-platform\\resource\\AC_26_3 booths.xlsx"
file_1 = "C:\\Users\\praga\\Documents\\aXtr\\aXtr-electionCampagin\\genai-hyper-personalized-platform\\resource\\AC_90_SALEM SOUTH.xlsx"

df1 = pd.read_excel(file_2)
df2 = pd.read_excel(file_1)
def normalize_columns(df):
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df

df1 = normalize_columns(df1)
df2 = normalize_columns(df2)

print("AC_90 columns:", df1.columns.tolist())
print("AC_26 columns:", df2.columns.tolist())
# ---------------- COLUMN MAPS ----------------
# AC_90_Salem
map_ac90 = {
    "part_no": "part_no",
    "section_no": "section_no",
    "srno": "serial_no",
    "epic_id": "epic_id",
    "house_number": "house_no",
    "name_eng": "voter_name_en",
    "relative_name_eng": "relative_name_en",
    "relation": "relation",
    "age": "age",
    "gender": "gender",
    "dob": "dob",
    "mobile": "mobile"
}

map_ac26 = {
    "part_no": "part_no",
    "section_no": "section_no",
    "slnoinpart": "serial_no",
    "epic_no": "epic_id",
    "c_house_no": "house_no",
    "fm_name_en": "voter_name_en",
    "rln_fm_nm_en": "relative_name_en",
    "rln_type": "relation",
    "age": "age",
    "gender": "gender",
    "dob": "dob",
    "mobile_no": "mobile"
}
# ---------------- SELECT + RENAME ----------------
df1_clean = df1[list(map_ac90.keys())].rename(columns=map_ac90)
df2_clean = df2[list(map_ac26.keys())].rename(columns=map_ac26)

print("DF 90 Cleaned",df1_clean.columns)
print("DF 26 Cleaned",df2_clean.columns)
# ---------------- ADD CONTEXT ----------------
df1_clean["assembly_constituency"] = "AC_90_Salem_South"
df2_clean["assembly_constituency"] = "AC_26_3"

df1_clean["source_file"] = "AC_90_SALEM"
df2_clean["source_file"] = "AC_26_3_BOOTHS"

# ---------------- MERGE ----------------
merged_df = pd.concat([df1_clean, df2_clean], ignore_index=True)
print(merged_df.columns )
# ---------------- OPTIONAL CLEANUPS ----------------
# merged_df["gender"] = merged_df["gender"].str.upper().str.strip()
# merged_df["epic_id"] = merged_df["epic_id"].str.upper().str.strip()

# # ---------------- EXPORT ----------------
merged_df.to_csv("voters_master_clean.csv", index=False)

print("✅ Clean merge completed with correct columns. Output: voters_master_clean.csv")


AC_90 columns: ['assembly', 'section_no_eng', 'section_no', 'part_no', 'srno', 'epic_id', 'house_number', 'name_eng', 'relative_name_eng', 'name', 'relative_name', 'relation', 'age', 'gender', 'mobile', 'dob']
AC_26 columns: ['ac_no', 'part_no', 'section_no', 'slnoinpart', 'c_house_no', 'c_house_no_v1', 'fm_name_en', 'lastname_en', 'fm_name_v1', 'lastname_v1', 'rln_type', 'rln_fm_nm_en', 'rln_l_nm_en', 'rln_fm_nm_v1', 'rln_l_nm_v1', 'epic_no', 'gender', 'age', 'dob', 'mobile_no']
DF 90 Cleaned Index(['part_no', 'section_no', 'serial_no', 'epic_id', 'house_no',
       'voter_name_en', 'relative_name_en', 'relation', 'age', 'gender', 'dob',
       'mobile'],
      dtype='object')
DF 26 Cleaned Index(['part_no', 'section_no', 'serial_no', 'epic_id', 'house_no',
       'voter_name_en', 'relative_name_en', 'relation', 'age', 'gender', 'dob',
       'mobile'],
      dtype='object')
Index(['part_no', 'section_no', 'serial_no', 'epic_id', 'house_no',
       'voter_name_en', 'relative_name_en',

In [40]:

df = merged_df.copy()

# Rename to match Supabase schema
df = df.rename(columns={
    "epic_id": "voter_id",
    "voter_name_en": "voter_name",
    "relative_name_en": "father_or_husband_name",
    "house_no": "door_no",
    "part_no": "booth_number"
})

# Columns that will go directly into Supabase
DIRECT_COLUMNS = [
    "voter_id",
    "voter_name",
    "father_or_husband_name",
    "gender",
    "age",
    "door_no",
    "booth_number",
    "assembly_constituency",
    "source_file"
]

# Everything else → raw_data
df["raw_data"] = df.drop(columns=DIRECT_COLUMNS, errors="ignore") \
                     .to_dict(orient="records")

# Keep only Supabase columns
df_final = df[DIRECT_COLUMNS + ["raw_data"]]

print(df_final.head())
df_final.to_csv("voters_master_clean.csv", index=False)

     voter_id      voter_name father_or_husband_name  gender  age door_no  \
0  XKR0630483   Karthikeyan               Shanmugam     Male   46     1     
1  XKR0618702    Apduljafar              Kamartheen     Male   45     1     
2  XKR2573673   Dineshkumar P             Paranjothi    Male   39     1     
3  XKR0499806  Balkees Beevi             Abduljafar   Female   36     1     
4  XKR0629766   Muthulakshmi            Karthikeyan   Female   33     1     

   booth_number assembly_constituency  source_file  \
0             1     AC_90_Salem_South  AC_90_SALEM   
1             1     AC_90_Salem_South  AC_90_SALEM   
2             1     AC_90_Salem_South  AC_90_SALEM   
3             1     AC_90_Salem_South  AC_90_SALEM   
4             1     AC_90_Salem_South  AC_90_SALEM   

                                            raw_data  
0  {'section_no': '1-வேளாச்சேரி வார்ட் நோ.௧௭௫ பெர...  
1  {'section_no': '1-வேளாச்சேரி வார்ட் நோ.௧௭௫ பெர...  
2  {'section_no': '1-வேளாச்சேரி வார்ட் நோ.௧௭௫ ப

In [30]:
import random
import pandas as pd

SECTION_MAP = {
    1: {
        "ward_no": 175,
        "area": "Velachery",
        "nagar": "Periyar Nagar",
        "street": "1st Street"
    },
    2: {
        "ward_no": 175,
        "area": "Velachery",
        "nagar": "Periyar Nagar",
        "street": "1st Cross Street"
    },
    3: {
        "ward_no": 175,
        "area": "Velachery",
        "nagar": "Periyar Nagar",
        "street": "2nd Street"
    },
    4: {
        "ward_no": 175,
        "area": "Velachery",
        "nagar": "Periyar Nagar",
        "street": "3rd Street"
    }
}


In [55]:
import random

# Chennai district localities (edit/extend this master list anytime)
CHENNAI_LOCALITIES = [
    "Velachery", "Madipakkam", "Adambakkam", "Ullagaram", "Nanganallur",
    "Keelkattalai", "Pallikaranai", "Perungudi", "Thoraipakkam", "Taramani",
    "Kottivakkam", "Neelankarai", "Sholinganallur", "Thiruvanmiyur", "Besant Nagar",
    "Guindy", "Alandur", "Saidapet", "Ashok Nagar", "KK Nagar",
    "Mylapore", "T Nagar", "Nungambakkam", "Egmore", "Triplicane",
    "Royapettah", "Teynampet", "Kodambakkam", "West Mambalam", "Vadapalani",
    "Virugambakkam", "Valasaravakkam", "Porur", "Kundrathur", "Pammal",
    "Chromepet", "Pallavaram", "Tambaram", "Perungalathur", "Medavakkam"
]




In [59]:
import pandas as pd
import random

# Load your Excel file
df = pd.read_csv("voters_with_random_locations.csv")  
# ⬆️ change filename to your actual file

# Ensure columns exist (create if missing)
if "village" not in df.columns:
    df["village"] = None

if "district" not in df.columns:
    df["district"] = None

# Update District → Chennai
df["district"] = "Chennai"

# Randomly assign Chennai villages (row-wise)
df["village"] = [random.choice(CHENNAI_LOCALITIES) for _ in range(len(df))]

# Save back to Excel
df.to_csv("location_updated_chennai.csv", index=False)

print("✅ Village and District fields updated successfully")


✅ Village and District fields updated successfully


In [47]:
df = pd.read_csv("voters_master_clean.csv")


In [48]:
df.columns

Index(['voter_id', 'voter_name', 'father_or_husband_name', 'gender', 'age',
       'door_no', 'booth_number', 'assembly_constituency', 'source_file',
       'raw_data'],
      dtype='object')

In [49]:
def assign_random_location():
    section = random.choice(list(SECTION_MAP.keys()))
    data = SECTION_MAP[section]

    return pd.Series({
       
        "ward": data["ward_no"],
        "area": data["area"],
        "street": data["street"]
    })


In [50]:
df[[ "ward", "area", "street"]] = df.apply(
    lambda _: assign_random_location(),
    axis=1
)


In [51]:
df.columns

Index(['voter_id', 'voter_name', 'father_or_husband_name', 'gender', 'age',
       'door_no', 'booth_number', 'assembly_constituency', 'source_file',
       'raw_data', 'ward', 'area', 'street'],
      dtype='object')

In [54]:
df.to_csv("voters_with_random_locations.csv", index=False)

print("✅ Random location assignment completed")

✅ Random location assignment completed


In [66]:
ISSUE_CATEGORY_TO_DESCRIPTIONS = {
    "Flooding & Drainage": [
        "Chronic flooding during NE monsoon",
        "Drainage overflow into homes during rains"
    ],
    "Water Supply & Groundwater": [
        "Irregular metro water supply in non-OMR zones",
        "Groundwater depletion & salinity intrusion",
        "Water disputes between RWAs & metro water"
    ],
    "Sewage, Sanitation & Waste Management": [
        "Velachery Lake encroachment & sewage inflow",
        "Waste management failures in apartment clusters",
        "Poor solid waste processing at Perungudi dump",
        "Inconsistent garbage collection in new layouts"
    ],
    "Transportation, Traffic & Connectivity": [
        "Traffic gridlock at Velachery–Taramani Link Road (VTLR)",
        "Lack of last-mile connectivity for IT professionals",
        "Overburdened Velachery Railway Station"
    ],
    "Road, Pedestrian & Public Safety": [
        "Pedestrian & cyclist fatalities on OMR",
        "Two-wheeler thefts near metro stations & IT parks",
        "Inadequate street lighting in IT corridor peripheries",
        "Lack of women’s safety infrastructure (CCTV, panic buttons)"
    ],
    "Power & Urban Infrastructure": [
        "Power cuts during summer peak (Apr–Jun)",
        "No dedicated EV charging infrastructure"
    ],
    "Housing, Rentals & Urban Cost of Living": [
        "Soaring rental & property prices",
        "Lack of affordable rental housing for service staff"
    ],
    "Health, Education & Social Infrastructure": [
        "Inadequate government schools & colleges",
        "Lack of primary healthcare centers (PHCs)",
        "Inadequate anganwadis in resettlement colonies"
    ],
    "Environment, Pollution & Public Spaces": [
        "Air & noise pollution from OMR traffic",
        "Lack of public parks & recreational spaces",
        "Mosquito breeding due to stagnant water"
    ],
    "Employment, Skills & Digital Access": [
        "Youth unemployment despite IT hub presence",
        "Digital divide in non-IT zones (e.g., slums near Perungudi)",
        "Youth demand for AI/skill centers near OMR"
    ]
}


In [67]:
VOTER_CATEGORY_TO_ISSUES = {
    "Youth / First-time Voter": [
        "Employment, Skills & Digital Access",
        "Health, Education & Social Infrastructure"
    ],
    "Working Professional": [
        "Transportation, Traffic & Connectivity",
        "Housing, Rentals & Urban Cost of Living",
        "Power & Urban Infrastructure"
    ],
    "Women": [
        "Road, Pedestrian & Public Safety",
        "Health, Education & Social Infrastructure"
    ],
    "Senior Citizens": [
        "Health, Education & Social Infrastructure",
        "Water Supply & Groundwater"
    ],
    "Daily Wage / Service Worker": [
        "Housing, Rentals & Urban Cost of Living",
        "Transportation, Traffic & Connectivity"
    ]
}


In [62]:
import pandas as pd
import random


In [ ]:
INPUT_CSV = "location_updated_chennai.csv"
OUTPUT_CSV = "voters_master.csv"

df = pd.read_csv(INPUT_CSV)


In [64]:
def assign_voter_category(row):
    age = row.get("age")
    gender = str(row.get("gender", "")).strip().lower()

    if pd.isna(age):
        return "Unknown"

    if age <= 21:
        return "Youth / First-time Voter"

    if age >= 60:
        return "Senior Citizens"

    if gender == "female":
        return "Women"

    if 22 <= age <= 45:
        return "Working Professional"

    return "Daily Wage / Service Worker"


In [68]:
def assign_issue_pipeline(row):
    voter_category = assign_voter_category(row)

    possible_issue_categories = VOTER_CATEGORY_TO_ISSUES.get(
        voter_category, []
    )

    if not possible_issue_categories:
        return pd.Series({
            "voter_category": voter_category,
            "issue_category": None,
            "issue_description": None
        })

    issue_category = random.choice(possible_issue_categories)
    issue_description = random.choice(
        ISSUE_CATEGORY_TO_DESCRIPTIONS[issue_category]
    )

    return pd.Series({
        "voter_category": voter_category,
        "issue_category": issue_category,
        "issue_description": issue_description
    })


In [69]:
df[["voter_category", "issue_category", "issue_description"]] = df.apply(
    assign_issue_pipeline,
    axis=1
)


In [70]:
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Issue mapping completed. Output saved to: {OUTPUT_CSV}")


✅ Issue mapping completed. Output saved to: voters_with_issues.csv
